<a href="https://colab.research.google.com/github/Amitabh-Phule/GenAi/blob/main/Exp4_GenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Design and development of a Retrieval-Augmented Generation (RAG) based question answering chat system.**
#Objectives:
1. To understand RAG architecture
2. To build a retrieval-based QA chatbot

# 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface langchain-groq faiss-cpu pypdf

# 2. Import Libraries

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq
from google.colab import files, userdata

# 3. Load GROQ API Key

In [ ]:
GROQ_API_KEY = userdata.get('Lab')

# 4. Upload PDF Document

In [ ]:
print("Upload your PDF:")
uploaded = files.upload()

for fn in uploaded.keys():
    pdf_path = fn

print(f"Loaded file: {pdf_path}")

# 5. Load PDF Content

In [ ]:
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"Total pages: {len(documents)}")

# 6. Text Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
texts = text_splitter.split_documents(documents)
print(f"Total chunks: {len(texts)}")

# 7. Embeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# 8. Vector Store (FAISS)

In [ ]:
vector_store = FAISS.from_documents(texts, embeddings)

# 9. Load GROQ LLM

In [ ]:
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

# 10. Retrieval QA Chain

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_store.as_retriever()
)

# 11. Chat Loop

In [ ]:
print("\n✅ RAG Chatbot Ready (type 'exit' to stop)\n")

while True:
    query = input("You: ")
    if query.lower() == "exit":
        print("Exiting chatbot...")
        break

    response = qa_chain.run(query)
    print("Bot:", response)